In [5]:
from langchain.llms import OpenAI
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate

def get_weather(lon, lat):
    print("call an api...")

function = {
    "name": "get_weather",
    "description":"function that takes longitude and latitude to find the weather of a place",
    "parameters":{
        "type":"object",
        "properties":{
            "lon":{
                "type":"string",
                "description":"The longitude coordinate"
            },
            "lat":{
                "type":"string",
                "description":"The latitude coordinate"
            }
        }
    },
    "required":["lon", "lat"],
}

llm = ChatOpenAI(
    temperature=0.1
).bind(
    function_call="auto",
    functions=[function]
)

prompt = PromptTemplate.from_template("Who is the weather in {city}?")

chain = prompt | llm

response = chain.invoke({"city":"rome"})
response = response.additional_kwargs["function_call"]["arguments"]

response

'{"lon":"12.4964","lat":"41.9028"}'

In [6]:
import json
r = json.loads(response)

get_weather(r["lon"], r["lat"])

call an api...


In [7]:
from bs4 import BeautifulSoup
import requests
import json

code = "272210"

url_target = "https://finance.naver.com/item/main.naver?code=" + code
print(url_target)
response = requests.get(url_target)
    #st.write(response.text)

soup = BeautifulSoup(response.text, 'html.parser')
result = soup.select_one('#middle > dl > dd:nth-child(5)')
result.text


https://finance.naver.com/item/main.naver?code=272210


'현재가 19,140 전일대비 보합 0  0.00 퍼센트'

In [8]:
import subprocess
from pydub import AudioSegment

def extract_audio_from_video(video_path, audio_path):
    ## ffmpeg -i files/podcast.mp4 -vn files/audio.mp3
    command = ["ffmpeg", "-i", video_path, "-vn", audio_path]
    subprocess.run(command)

#extract_audio_from_video("files/podcast.mp4", "files/podcast.mp3")

track = AudioSegment.from_mp3("./files/podcast.mp3")

In [9]:
ten_minutes = 60*10*1000
first_five = track[:ten_minutes]
first_five.export("./files/first_five.mp3", format="mp3")

<_io.BufferedRandom name='./files/first_five.mp3'>

In [11]:
import math

chunks = math.ceil(len(track) / ten_minutes)

for i in range(chunks):
    start_time = i * ten_minutes
    end_time = (i + 1) * ten_minutes

    chunk = track[start_time:end_time]

    chunk.export(f"./files/chunks/chunk_{i}.mp3", format="mp3")

In [ ]:
def cut_audio_in_chunks(audio_path, chunk_size, chunks_folder):
    track = AudioSegment.from_mp3(audio_path)
    chunk_len = chunk_size * 60 * 1000 
    chunks = math.ceil(len(track) / chunk_len)

    for i in range(chunks):
        start_time = i * chunk_len
        end_time = (i + 1) * chunk_len

    chunk = track[start_time:end_time]

    chunk.export(f"./{chunks_folder}/chunk_{i}.mp3", format="mp3")

cut_audio_in_chunks("./files/podcast.mp3, 10, ./files/chunks")



In [12]:
import openai

transcript = openai.Audio.transcribe(
    "whisper-1", open("./files/chunks/chunk_0.mp3", "rb"),
)

transcript

<OpenAIObject at 0x22499f520f0> JSON: {
  "text": "If success is this lagging indicator of commitment now, how can you be sure that you are paying your dues? The best-selling author and host. The number one health and wellness podcast. On Purpose with Jay Shetty. Society has gone in the direction of becoming addicted to pleasure. Yes. Or pleasure-seeking. Where, from the Stoic's perspective, why did we even ever go down that road? Like, why did we leave wisdom and self-control? Or did we never have it at all and we've always been trying to balance it? Yeah. I mean, I guess that's the big question is like, why do we take something that we like too far? Yeah. Right. So the Epicureans would say like, look, drinking is great, but if you have a hangover the next day, was it actually so great? And so, you know, if you, if you push the pleasure too far, it becomes not pleasurable, but in the moment that feels very far away, right? Like in the moment you want the thing now. Obviously sex is th

In [14]:
import glob

def transcripbe_chunks(chunk_folder, destination):
    files = glob.glob(f"{chunk_folder}/*.mp3")
    final_transcript = ""
    for file in files :
        with open(file, "rb") as audio_file:
            transcript = openai.Audio.transcribe(
                "whisper-1", open(file, "rb"),
             )   
            final_transcript += transcript["text"]

    with open(destination, "w") as file:
        file.write(final_transcript)

    return final_transcript

transcripbe_chunks("./files/chunks", "./files/transcript.txt")

"If success is this lagging indicator of commitment now, how can you be sure that you are paying your dues? The best-selling author and host. The number one health and wellness podcast. On Purpose with Jay Shetty. Society has gone in the direction of becoming addicted to pleasure. Yes. Or pleasure-seeking. Where, from the Stoic's perspective, why did we even ever go down that road? Like, why did we leave wisdom and self-control? Or did we never have it at all and we've always been trying to balance it? Yeah. I mean, I guess that's the big question is like, why do we take something that we like too far? Yeah. Right. So the Epicureans would say like, look, drinking is great, but if you have a hangover the next day, was it actually so great? And so, you know, if you, if you push the pleasure too far, it becomes not pleasurable, but in the moment that feels very far away, right? Like in the moment you want the thing now. Obviously sex is this thing for people. It's like the thing you're at